# 02 — Validation des données et contrats

**Projet** : Espace vectoriel creux d'un catalogue dupliqué (TF-IDF cosinus) — la référence à battre

## Objectifs pédagogiques

1. Lire les contrats du projet : documents, questions, passages et prédictions.
1. Vérifier que le corpus réel les respecte, puis observer ce qui se passe quand un fichier ne les respecte plus.
1. Comprendre la règle « question hors corpus ⇔ extrait vide », qui empêche une annotation incohérente de gonfler le rappel.

**Objectifs transverses du dépôt**

- Construire un index vectoriel déterministe et hors ligne : hachage de n-grammes, SVD tronquée apprise sur le train, normalisation L2.
- Mesurer les propriétés d'un index au lieu de les supposer : dimension réellement produite, fidélité de la projection, taille de l'artefact.
- Chiffrer ce que la compression apporte et ce qu'elle coûte : 128 dimensions contre 4096, 0,16 Mo de matrice contre 5,25 Mo, fidélité de projection de 1,00.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (64 documents) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 64

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

CHUNKING = dict(node(CONFIG, "preprocessing").get("chunking", {}))

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (plancher contractuel : 0.9)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(
    f"Découpage         : {CHUNKING.get('max_tokens')} tokens, "
    f"chevauchement {CHUNKING.get('overlap_tokens')}"
)
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

## 1. Les contrats

Un contrat de données est du code exécutable : il nomme chaque colonne, son type et ses règles croisées. C'est aussi la documentation la plus fiable du jeu de données, puisqu'elle est vérifiée à chaque exécution du pipeline.

In [ ]:
from src.data.schemas import (
    ChunksSchema,
    DocumentsSchema,
    PredictedAnswersSchema,
    QueriesSchema,
    validate_documents,
    validate_queries,
)

for schema in (DocumentsSchema, QueriesSchema, ChunksSchema, PredictedAnswersSchema):
    print(f"{schema.__name__:22s} -> {', '.join(schema.to_schema().columns)}")

### 1.1 Le corpus respecte ses contrats

In [ ]:
from src.data.generators import SyntheticCorpusGenerator
from src.data.loaders import TextCorpusLoader

LOADER = TextCorpusLoader(PATHS, formats=CONFIG.data.formats)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    QUERIES = LOADER.load_queries()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} documents, {len(QUERIES)} questions")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, QUERIES, METADATA = bundle.documents, bundle.queries, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_queries(QUERIES)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} documents généré et persisté")

CALIBRATION = QUERIES[QUERIES["split"] == "calibration"].reset_index(drop=True)
VALIDATION = QUERIES[QUERIES["split"] == "val"].reset_index(drop=True)
TEST = QUERIES[QUERIES["split"] == "test"].reset_index(drop=True)
ANSWERABLE = QUERIES[QUERIES["answer_type"] != "unanswerable"].reset_index(drop=True)
# Sous-ensemble utilisé pour **régler** quoi que ce soit : la validation seule. Le test ne sert
# qu'à juger, jamais à choisir (y compris dans un notebook).
VAL_ANSWERABLE = VALIDATION[VALIDATION["answer_type"] != "unanswerable"].reset_index(drop=True)
print(
    "splits :",
    {
        name: len(frame)
        for name, frame in (("calibration", CALIBRATION), ("val", VALIDATION), ("test", TEST))
    },
)


validated_documents = validate_documents(DOCUMENTS)
validated_queries = validate_queries(QUERIES)
print(f"{len(validated_documents)} documents valides, {len(validated_queries)} questions valides")

# La convention est explicite : une question hors corpus désigne un document leurre (celui qui en
# est le plus proche) mais **aucun extrait** — c'est l'extrait qui porte la réponse. Le contrat
# vérifie que les deux marqueurs sont d'accord : « hors corpus » <=> « extrait vide ».
unanswerable = validated_queries["answer_type"] == "unanswerable"
empty_span = validated_queries["gold_span"].str.strip() == ""
assert bool((unanswerable == empty_span).all())
print("règle vérifiée : answer_type == 'unanswerable' <=> extrait annoté vide")
print(
    "questions hors corpus désignant tout de même un document leurre :",
    int((unanswerable & (validated_queries["gold_doc_ids"] != "")).sum()),
    "-> leur pertinence est jugée nulle, c'est l'extrait qui compte",
)

## 2. Casser volontairement les contrats

Chaque corruption ci-dessous correspond à un incident réel : identifiant mal formé, source inconnue, colonne manquante, question annotée sans document. Le message d'erreur de Pandera est affiché tel quel — c'est lui qu'on lira en production.

In [ ]:
from typing import Any


def show_failure(label: str, frame: pd.DataFrame, validator: Any) -> None:
    """Afficher la première ligne du message d'erreur d'un contrat."""
    try:
        validator(frame)
    except Exception as error:  # on veut le message du contrat, quel qu'il soit
        print(f"[refusé] {label} : {str(error).splitlines()[0]}")
    else:
        print(f"[accepté] {label} : aucune erreur (le contrat est trop laxiste)")


bad_id = DOCUMENTS.head(1).copy()
bad_id.loc[bad_id.index[0], "doc_id"] = "doc-1"
show_failure("identifiant hors format", bad_id, validate_documents)

bad_source = DOCUMENTS.head(1).copy()
bad_source.loc[bad_source.index[0], "source"] = "wiki_inconnu"
show_failure("source inconnue", bad_source, validate_documents)

missing_column = DOCUMENTS.drop(columns=["text"])
show_failure("colonne manquante", missing_column, validate_documents)

In [ ]:
answerable = ANSWERABLE.head(1).copy()
answerable.loc[answerable.index[0], "gold_doc_ids"] = ""
show_failure("question répondable sans document annoté", answerable, validate_queries)

sabotaged = QUERIES[QUERIES["answer_type"] == "unanswerable"].head(1).copy()
sabotaged.loc[sabotaged.index[0], "gold_span"] = "Un extrait qui n'existe pas."
show_failure("question hors corpus avec extrait", sabotaged, validate_queries)

In [ ]:
# Le mode *lazy* rassemble toutes les violations en une seule erreur : c'est ce qu'on veut dans un
# rapport de qualité de données, où l'on préfère corriger plusieurs défauts en un seul passage.
broken = DOCUMENTS.head(3).copy()
broken.loc[broken.index[0], "doc_id"] = "nope"
broken.loc[broken.index[1], "source"] = "wiki_inconnu"
try:
    validate_documents(broken, lazy=True)
except Exception as error:
    cases = getattr(error, "failure_cases", None)
    print("violations détectées :", 0 if cases is None else len(cases))
    if cases is not None:
        display(cases)

**Ce qu'il faut retenir**

- Un contrat protège la **jointure** : identifiants, catégories, cohérence croisée. Sans lui, une question mal annotée se traduit par un rappel nul dont personne ne retrouve l'origine.
- Le mode strict refuse toute colonne non déclarée : un fichier personnalisé ne peut pas injecter silencieusement une colonne dans le corpus publié.
- La règle « hors corpus ⇔ extrait vide » est la seule qui distingue proprement « je ne sais pas » de « je réponds n'importe quoi ».